# Causal Uplift Modeling & Next-Gen Decision Intelligence
## Final Year Honors Project in Data Science

### Abstract
Traditional predictive machine learning models conflate correlation with causation by estimating $P(Y=1 | X)$. In retention and promotion campaigns, this leads to significant budget waste on **Sure Things** (organic buyers) and churn acceleration among **Sleeping Dogs**.

This notebook demonstrates an end-to-end causal machine learning workflow estimating the **Conditional Average Treatment Effect (CATE)**:
$$\tau(X) = \mathbb{E}[Y(1) - Y(0) \mid X]$$

**Pipeline Contents:**
1. Experimental Data Ingestion (64,000 customers from MineThatData benchmark)
2. Causal Diagnostics: Positivity (Common Support) & Covariate Balance (SMD)
3. Meta-Learner Implementations (S-Learner, T-Learner, X-Learner)
4. Counterfactual Evaluation: Qini Curves, AUUC, and Uplift Deciles
5. Split Conformal Prediction for 90% Uncertainty Risk Bounds
6. Multi-Offer Economic Optimization (10% vs 20% vs Free Shipping)

In [ ]:
import sys
import os
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import download_or_generate_dataset, preprocess_data, prepare_train_test_split
from src.causal_validation import run_full_causal_diagnostics
from src.meta_learners import train_all_learners
from src.evaluation import calculate_qini_curve, compute_uplift_deciles, simulate_business_roi
from src.conformal_uplift import get_default_conformal_predictor
from src.multi_treatment_optimizer import MultiTreatmentOptimizer

## 1. Data Ingestion & Causal Diagnostics

In [ ]:
raw_df = download_or_generate_dataset('../data/raw/minethatdata_email.csv')
print(f"Dataset Shape: {raw_df.shape}")
raw_df.head()

In [ ]:
X, treatment, outcome = preprocess_data(raw_df)
diag = run_full_causal_diagnostics(X, treatment)
print(f"Common Support: {diag['common_support']['percent_in_common_support']:.1f}%")
print(f"Max SMD: {diag['max_smd']:.4f}")
diag['smd_table'].head()

## 2. Meta-Learner Training & Benchmarking

In [ ]:
splits = prepare_train_test_split(X, treatment, outcome, test_size=0.3, seed=42)
models = train_all_learners(splits['X_train'], splits['treatment_train'], splits['outcome_train'])

## 3. Qini Curve & Uplift Deciles Evaluation

In [ ]:
y_test = splits['outcome_test'].values
tr_test = splits['treatment_test'].values

plt.figure(figsize=(8, 5))
for name, model in models.items():
    preds = model.predict_uplift(splits['X_test'])
    qini = calculate_qini_curve(y_test, tr_test, preds)
    plt.plot(qini['fractions'], qini['qini_values'], label=f"{name} (Qini={qini['qini_score']:.2f})")

plt.plot([0, 1], [0, qini['random_values'][-1]], 'k--', label="Random Targeting Baseline")
plt.xlabel("Fraction of Population Targeted")
plt.ylabel("Cumulative Incremental Conversions")
plt.title("Qini Curve Benchmark Comparison")
plt.legend()
plt.grid(True)
plt.show()

## 4. Conformal Uncertainty Bounds & Multi-Offer Optimization

In [ ]:
conformal_engine = get_default_conformal_predictor()
multi_opt = MultiTreatmentOptimizer(customer_ltv=80.0)

# Sample customer
sample_customer = splits['X_test'].iloc[0].to_dict()
tau_hat = models['T-Learner'].predict_uplift(splits['X_test'].iloc[[0]])[0]

ci = conformal_engine.predict_intervals(tau_hat)
print("90% Conformal Interval:", ci)

offer_decision = multi_opt.optimize_offer(sample_customer, base_tau=tau_hat)
print("Optimal Offer Prescribed:", offer_decision['optimal_offer'])